In [1]:
# !pip install crochet

In [1]:
import json

location_json = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\input_json\bbox_indo.json'


In [37]:
base_url = 'https://geoportal.menlhk.go.id/server/rest/services/Time_Series/PL_2020/MapServer/0/query'

# scrape webpage
import scrapy
from scrapy.crawler import CrawlerRunner

from urllib.parse import urlencode

# Reactor restart
from crochet import setup, wait_for
setup()

class SpiderTest(scrapy.Spider):
    name = 'testing_rest_api'
    start_urls = [base_url]

    def parse(self, response):
        
        #oid_name = response.meta.get('oid_name')
        layer_name = 'PL_2020'
        layer_link_name = '/server/rest/services/Time_Series/PL_2020/MapServer/0'
        with open(location_json, 'r') as json_file:
        # with open('./input_json/bbox.json', 'r') as json_file:
            dict_data = json.load(json_file)

        str_bbox = str(dict_data)
        print(str_bbox)

        params = {
                "where": "objectid >= -1",
                "text": "",
                "objectIds": "",
                "time": "",
                "timeRelation": "esriTimeRelationOverlaps",
                "geometry": str_bbox,
                "geometryType": "esriGeometryEnvelope",
                "inSR": 4326,
                "spatialRel": "esriSpatialRelIntersects",
                "units": "esriSRUnit_Foot",
                "outFields": "*",
                "returnGeometry": "false",
                "returnTrueCurves": "false",
                "maxAllowableOffset": "",
                "geometryPrecision": "",
                "outSR": 4326,
                "havingClause": "",
                "returnIdsOnly": "true",
                "returnCountOnly": "false",
                "orderByFields": "",
                "groupByFieldsForStatistics": "",
                "outStatistics": "",
                "returnZ": "false",
                "returnM": "false",
                "gdbVersion": "",
                "historicMoment": "",
                "returnDistinctValues": "false",
                "resultOffset": "",
                "resultRecordCount": "",
                "returnExtentOnly": "false",
                "sqlFormat": "none",
                "datumTransformation": "",
                "parameterValues": "",
                "rangeValues": "",
                "quantizationParameters": "",
                "featureEncoding": "esriDefault",
                "f": "geojson"
            }

        page_url = 'https://geoportal.menlhk.go.id' + layer_link_name + '/query?' + urlencode(params)
        #page_url = 'https://geoportal.menlhk.go.id' + layer_link_name + '/query'

        yield scrapy.Request(page_url, self.parse_query_post, meta={'layer_link_name':layer_link_name})

    def parse_query_post(self, response):
        #output_dir = './output_json'  # Define your output directory
        #os.makedirs(output_dir, exist_ok=True)
        print(response.text)

        layer_link_name = response.meta.get('layer_link_name')

        layer_name_parts = layer_link_name.split('/')
        layer_name = layer_name_parts[-3]
        
        

        data = json.loads(response.text)

        dict_oid = {layer_name: data}
        print(dict_oid)

In [42]:
@wait_for(200)
def run_spider():
    crawler = CrawlerRunner()
    d = crawler.crawl(SpiderTest)
    return d

In [43]:
# Run the spider #NOT EFFECTIVE TO RUN SCRAPY IN JUPYTER NOTEBOOK!
run_spider()

{'xmin': 90.00000000000006, 'ymin': -14.999999999999943, 'xmax': 144.0000000000001, 'ymax': 8.000000000000057}


TimeoutError: 